you need to install some Python libraries that will help you with cryptographic functions and data encoding. The key libraries used are:

ecdsa: Used for elliptic curve cryptography to generate private and public keys.

hashlib: Provides hashing algorithms such as SHA-256, crucial for creating addresses.

base58: Encoding used for generating the final wallet address format.

In [ ]:
!pip install ecdsa base58 pycryptodome

# Step 1: Generate a Private Key
With every application in Python, the frist thing you need to do for your wallet application is to generate a private key, a 256-bit number that will be used to sign transactions and access funds. In Python, we can use the os.urandom() function to create this random 32-byte private key.

In [ ]:
import os

private_key = os.urandom(32)
print(f"Private Key: {private_key.hex()}")

Private Key: ba9e71ebeb8bda16daa114c50e949bb9e006f702cd24a788f508adfe626aa49c


# Step 2: Generate the Public Key
Once you’ve generated the private key, you can derive the public key. The public key is created using elliptic curve cryptography, specifically the ecdsa (Elliptic Curve Digital Signature Algorithm) library in Python.

We use the SECP256k1 curve, which is widely used in Bitcoin and other cryptocurrencies:

In [ ]:
import ecdsa

sk = ecdsa.SigningKey.from_string(private_key, curve=ecdsa.SECP256k1)
vk = sk.get_verifying_key()
public_key = b"\x04" + vk.to_string()  # Adding 0x04 byte to signify uncompressed public key
print(f"Public Key: {public_key.hex()}")

Public Key: 043d90605d95c60dfbd816779f4b2f42d935d4991ee42cf9ba3c911459fb4f77cabeb072fa6076dc36f63e775f1bd0c5b92503542f961a45eb26a9b0b4ad1da3aa


# Step 3: Generate the Wallet Address
Now that you’ve acquired a public key, you can generate the wallet address. To do this, you’ll apply several hashing algorithms to the public key and format it in a way that can be used as a wallet address.

The process involves hashing the public key with SHA-256, then applying RIPEMD-160 for further compression, and finally encoding the result with Base58.

In [ ]:
import hashlib
import base58
from Cryptodome.Hash import RIPEMD160

# Step 1: SHA-256 Hash of Public Key
sha256_hash = hashlib.sha256(public_key).digest()

# Step 2: RIPEMD-160 Hash of SHA-256
# Use Cryptodome.Hash.RIPEMD160 for ripemd160 hashing
ripemd160_hasher = RIPEMD160.new()
ripemd160_hasher.update(sha256_hash)
ripemd160 = ripemd160_hasher.digest()

# Step 3: Adding Network Byte (0x00 for Bitcoin)
network_byte = b'\x00' + ripemd160

# Step 4: Creating a Checksum by Double Hashing
checksum = hashlib.sha256(hashlib.sha256(network_byte).digest()).digest()[:4]

# Step 5: Adding Checksum to the Versioned RIPEMD-160 Hash
address = base58.b58encode(network_byte + checksum)
print(f"Wallet Address: {address.decode()}")

Wallet Address: 13s4dddJcXhUV64r7Sirr9SrF1LGR4Z6AU


In [ ]:
%%writefile crypto_wallet.py
import os
import ecdsa
import hashlib
import base58
from Cryptodome.Hash import RIPEMD160

# Generate a random 32-byte private key
private_key = os.urandom(32)
print(f"Private Key: {private_key.hex()}")

# Derive the public key
sk = ecdsa.SigningKey.from_string(private_key, curve=ecdsa.SECP256k1)
vk = sk.get_verifying_key()
public_key = b"\x04" + vk.to_string()  # Adding 0x04 byte for uncompressed public key
print(f"Public Key: {public_key.hex()}")

# Generate the Bitcoin wallet address
# Step 1: SHA-256 Hash of Public Key
sha256_hash = hashlib.sha256(public_key).digest()

# Step 2: RIPEMD-160 Hash of SHA-256
ripemd160_hasher = RIPEMD160.new()
ripemd160_hasher.update(sha256_hash)
ripemd160 = ripemd160_hasher.digest()

# Step 3: Adding Network Byte (0x00 for Bitcoin)
network_byte = b'\x00' + ripemd160

# Step 4: Creating a Checksum by Double Hashing
checksum = hashlib.sha256(hashlib.sha256(network_byte).digest()).digest()[:4]

# Step 5: Adding Checksum to the Versioned RIPEMD-160 Hash
address = base58.b58encode(network_byte + checksum)
print(f"Wallet Address: {address.decode()}")

Overwriting crypto_wallet.py


In [ ]:
!python crypto_wallet.py

Private Key: 2ed02688a70ce424e5463f26a728128abd8dcc67e4dc8bbc9a222498f637a446
Public Key: 046e05c9f2274857297f29f56798049bcb8b9c3381e063ab4f6d3f7a5d0fdab8fd0f58343de1a03d0650f8d5e8ab708bedfa22707a4dd5e273c97a58e232461400
Wallet Address: 1DgXENLxNPwwvAcqWrgpjRm7sa3T7taRh7


Here's the code to create the `crypto_wallet.py` file. This script will generate a new private key, public key, and Bitcoin wallet address each time it's run.